# 🪨 GeoPINN Studio — Colab GPU Backend

**GeoPINN Studio v3.0.0.2** — Beylikova REE-F-Ba-Th Yatağı Jeofizik Suite

Bu notebook'u çalıştırarak GeoPINN Studio masaüstü uygulamasına Colab GPU backend'i bağlayabilirsiniz.

### Gereksinimler
- Google hesabı (ücretsiz Colab)
- ngrok hesabı — [ngrok.com](https://ngrok.com) (ücretsiz)
- GeoPINN Studio masaüstü uygulaması — [GitHub Releases](https://github.com/bilaltlc/geopinn-studio/releases)

### Adımlar
1. **Çalışma zamanı → GPU seç** (T4 ücretsiz)
2. Hücreleri sırasıyla çalıştır
3. Oluşan ngrok URL'sini GeoPINN Studio Ayarlar'a gir
4. Analiz yap! 🚀

In [ ]:
# ── 1. GPU Kontrolü ──────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA mevcut: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠ GPU yok — Çalışma Zamanı → GPU seçin')

In [ ]:
# ── 2. Bağımlılıkları Kur ────────────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok python-multipart scipy numpy
!pip install -q simpeg discretize  # SimPEG Tikhonov inversion için
print('✓ Bağımlılıklar kuruldu')

In [ ]:
# ── 3. ngrok Token ───────────────────────────────────────────────────────────
# ngrok.com/signup → ücretsiz hesap → Auth Token kopyala
from pyngrok import ngrok

NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # ← buraya token'ı yapıştır

if NGROK_TOKEN == 'YOUR_NGROK_TOKEN_HERE':
    print('⚠ ngrok token girilmedi!')
    print('ngrok.com → Dashboard → Auth Token → kopyalayıp buraya yapıştır')
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✓ ngrok token ayarlandı')

In [ ]:
# ── 4. GeoPINN Backend'i Drive'dan Yükle ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# Drive'daki backend klasörü
BACKEND_PATH = '/content/drive/MyDrive/geopinn_data'

if not os.path.exists(BACKEND_PATH):
    print(f'⚠ {BACKEND_PATH} bulunamadı')
    print('Drive\'a geopinn_data/ klasörü oluşturup server.py ve engines/ klasörünü yükleyin')
    print('Alternatif: GitHub\'dan direkt klonlayın (aşağıdaki hücre)')
else:
    sys.path.insert(0, BACKEND_PATH)
    os.chdir(BACKEND_PATH)
    os.makedirs('uploads', exist_ok=True)
    print(f'✓ Backend yolu: {BACKEND_PATH}')
    print(f'Dosyalar: {os.listdir(BACKEND_PATH)}')

In [ ]:
# ── 4b. ALTERNATİF: GitHub'dan Klonla ───────────────────────────────────────
# Drive yoksa veya ilk kurulumsa bu hücreyi kullan

# !git clone https://github.com/bilaltlc/geopinn-studio.git /content/geopinn
# import os, sys
# BACKEND_PATH = '/content/geopinn/geopinn-backend'
# sys.path.insert(0, BACKEND_PATH)
# os.chdir(BACKEND_PATH)
# os.makedirs('uploads', exist_ok=True)
# print('✓ GitHub\'dan klonlandı')

In [ ]:
# ── 5. Server Başlat ─────────────────────────────────────────────────────────
import subprocess, threading, time, requests
from pyngrok import ngrok

# Eski tunnel'ları kapat
ngrok.kill()
time.sleep(1)

# uvicorn başlat
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND_PATH,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(4)

# ngrok tunnel aç
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

# Sağlık kontrolü
try:
    r = requests.get(f'http://localhost:8000/api/health', timeout=5)
    info = r.json()
    print(f'✓ Server v{info["version"]} hazır')
    print(f'✓ GPU: {info.get("gpu", "bilinmiyor")}')
except Exception as e:
    print(f'⚠ Health check başarısız: {e}')

print(f'''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🌐 GEOPINN STUDIO BAĞLANTI URL:

   {PUBLIC_URL}

GeoPINN Studio → ⚙ Ayarlar → Colab → Bu URL'yi gir
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')

In [ ]:
# ── 6. Endpoint Kontrolü ─────────────────────────────────────────────────────
import requests, json

endpoints = [
    '/api/health',
    '/api/simpeg/status',
    '/api/fvm/status',
    '/api/radiometry/status',
]

for ep in endpoints:
    try:
        r = requests.get(f'http://localhost:8000{ep}', timeout=3)
        d = r.json()
        status = '✓' if r.ok else '✗'
        avail = d.get('available', d.get('status', 'ok'))
        print(f'{status} {ep}: {avail}')
    except Exception as e:
        print(f'✗ {ep}: {e}')

In [ ]:
# ── 7. Oturum Canlı Tut (Opsiyonel) ─────────────────────────────────────────
# Colab 90 dakika sonra oturumu kapatır.
# Bu hücre her 10 dakikada bir ping atarak oturumu canlı tutar.
# NOT: Colab kullanım politikasına göre otomatik canlı tutma desteklenmeyebilir.

import time, threading

def keep_alive():
    while True:
        try:
            requests.get('http://localhost:8000/api/health', timeout=2)
        except:
            pass
        time.sleep(600)  # 10 dakika

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('✓ Keep-alive başlatıldı (10 dk ping)')

## Sorun Giderme

**ngrok 5 tunnel limiti hatası:**
```python
from pyngrok import ngrok; ngrok.kill()
```

**SimPEG yok hatası:**
```bash
!pip install simpeg discretize
```

**Drive bağlanamıyor:**  
Runtime → Restart runtime, sonra tekrar dene.

**GPU yok:**  
Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU

---
📧 Destek: telcihamdibilal@gmail.com  
🐙 GitHub: https://github.com/bilaltlc/geopinn-studio